In [ ]:
#| default_exp logger

# logger

> simple logger using idiomatic Solveit

In [ ]:
#| export
import asyncio, time
from fastcore.all import patch
from datetime import datetime
from anyio import sleep
from dialoghelper.core import update_msg, add_msg, del_msg, read_msg, find_msg_id
from dutil.core import waitpred, waitpreda

In [ ]:
import random
import fastcore.all as FC
from fastcore.test import *

In [ ]:
#| export
_notfound = {'msg': {}}
def findlogtarget(id:str='', n:int=1):
    print('aaaa')
    if (msg := read_msg(0, id=id) if id else read_msg(n)) == _notfound: return ''
    return msg.id if (cnt := msg.content).count('\n') == cnt.count('\u200b')-1 else ''

In [ ]:
test_eq(findlogtarget(read_msg().id), read_msg().id)

In [ ]:
test_eq(findlogtarget(), read_msg(1).id)

In [ ]:
test_eq(findlogtarget(n=-1), read_msg().id)

In [ ]:
#| export
class Logger:
    logs:list; msgid:str; _s:str
    def __init__(self, id:str|None=''): self.logs = []; self.setup(id, True)
    def setup(self, id:str|None='', clear:bool=False): 
        self.msgid = id
        if id or id is None:  # use print or reuse given target
            print(id)
            if clear: self.clear()
            return
        if id := findlogtarget():  # reuse or create a new target
            print('findlogtarget', id)
            self.msgid = id
            if clear: self.clear()
            return
        print ('new')
        self.msgid = add_msg('\u200b' if clear else self._s, msg_type='raw')
    def clear(self): 
        self.logs.clear(); self._s='\u200b'; 
        if self.msgid: update_msg(self.msgid, content=self._s, msg_type='raw')
    async def settle(self, to:float=2.0):
        if self.msgid:
            t0 = time.time()
            while read_msg(1).id != self.msgid: 
                if time.time()-t0 <= to: await asyncio.sleep(0.1)
    def __call__(self, msg, *args, **kwargs): 
        s = f"[{datetime.now():%H:%M:%S}] {msg}"
        self.logs.insert(0, s)
        if self.msgid:
            self._s = f"\u200b{s}\n" + self._s if self._s != '\u200b' else s
            update_msg(self.msgid, content=self._s)
        else: print(s, flush=True)


In [ ]:
log = Logger()
print(log.msgid)
# await log.settle()
# test_eq(read_msg(1).id, log.msgid)

aaaa
new
_d11e82ac


In [ ]:
log('test')
log('test2')
log('test3')

In [ ]:
log(s := ';qwedcv fjkds')
test_is(s in log.logs[0], True)

In [ ]:
log.logs

['[15:14:54] ;qwedcv fjkds',
 '[15:14:41] test3',
 '[15:14:41] test2',
 '[15:14:41] test']

In [ ]:
log.clear()

In [ ]:
log.logs

[]

In [ ]:
log(f"Some msg {random.randint(0, 1000)}")

In [ ]:
log.logs

['[15:15:12] Some msg 566']

In [ ]:
#| export
@patch
def show(self:Logger, clear:bool=False, replace:bool=False):
    if not self.msgid or read_msg(1) == self.msgid: return
    oldid = self.msgid
    self.setup(find_msg_id() if replace else findlogtarget(), clear)
    if oldid != self.msgid: del_msg(self.msgid)
    time.sleep(0.2)
    if self.msgid and not clear: update_msg(id=self.msgid, msg_type='raw', content=self._s)

In [ ]:
if (msgid := findlogtarget()) and msgid != log.msgid: del_msg(msgid)
print(read_msg(1).id != msgid)
log.show()

In [ ]:
add_msg(content='\u200b[09:53:00] some other msg\n\u200b[09:44:16] some msg\n', id=read_msg(1).id, msg_type='raw');

In [ ]:
log.show()
await waitpreda(lambda: read_msg(1).id == log.msgid);

In [ ]:
log.show()

In [ ]:
# log.show(replace=True)  # duplicate, uncomment and run 

Look at `show` method. What it does?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The `show` method relocates the logger's output message in the dialog:

1. Deletes the current log message (at `self.msgid`)
2. Calls `setup()` to create a new message at a different location:
   - If `replace=True`: uses `find_msg_id()` (current message location)
   - Otherwise: uses `findlogtarget()` to find the previous raw message that looks like a log (has `\u200b` markers matching newlines)
3. Waits 0.2 seconds
4. Updates the new message with the log content (`self._s`)
5. If `clear=True` is passed, it also clears the log history

So it essentially moves the log output to either replace the current message or append to an existing log message found above in the dialog.

Let's create both versions. First, the async version:

In [ ]:
@patch
async def ashow(self:Logger, clear:bool=False, replace:bool=False, timeout:float=2.0):
    if not self.msgid: return
    oldid = self.msgid
    target_id = find_msg_id() if replace else findlogtarget()
    self.setup(target_id, clear)
    if oldid != self.msgid: del_msg(oldid)
    # n, t0 = -1 if replace else 1, time.time()
    # while findlogtarget(n) != self.msgid:
    #     if time.time() - t0 > timeout: return
    #     await sleep(0.1)
    n = 0 if replace else 1
    await waitpreda(lambda: findlogtarget(n) != self.msgid, timeout=timeout)
    if self.msgid: update_msg(id=self.msgid, msg_type='raw', content=self._s)

In [ ]:
# await log.ashow()

In [ ]:
# %quickref

In [ ]:
# %%timeit?

In [ ]:
@patch
def show(self:Logger, clear:bool=False, replace:bool=False, timeout:float=2.0):
    if not self.msgid: return
    oldid = self.msgid
    self.setup(find_msg_id() if replace else findlogtarget(), clear)
    if oldid != self.msgid: del_msg(oldid)
    # def _wait():
    #     n, t0 = -1 if replace else 1, time.time()
    #     while findlogtarget(n) != self.msgid:
    #         if time.time() - t0 > timeout: return
    #         time.sleep(0.1)
    # t = FC.threaded(_wait)(); t.join()
    n = 0 if replace else 1
    loc = msg_idx()
    waitpred(lambda: msg_idx(self.msgid) != loc+1, timeout=timeout)
    time.sleep(0.1)
    if self.msgid: update_msg(self.msgid, msg_type='raw', content=self._s)

In [ ]:
# log('asdf')

In [ ]:
log.show()
# print(read_msg(1).content, log._s)

In [ ]:
msg_idx(log.msgid)

In [ ]:
# update_msg(log.msgid, i_collapsed=True)

In [ ]:
log.logs

In [ ]:
msg_idx(log.msgid)

# export -

In [ ]:
from dutil.flakes import show_flakes
show_flakes()

In [ ]:
# #|hide
# #|eval: false
# import fastcore.all as FC
# from nbdev import nbdev_export
# if FC.IN_NOTEBOOK:
#     nb_path = '01_logger.ipynb'
#     nbdev_export(nb_path)